# Session 9 — From Q-tables to neural policies

**DAT 5668 · Module 5, Session 1 · Fundamentals (about 80 minutes)**

**Family this session: RL-based agent.** A neural Q-function is still an RL-based agent. It is not a replacement for the LLM-based agent that reads documents.

![Deep Q-network](../../assets/dqn.png)

## Run of show
- 10 min — When did the Week 4 table work? (Few states, one SKU.)
- 20 min — Function approximation, replay, target networks. Policy vs value vs model, only as a choice.
- 30 min — A tiny network Q-function and a replay buffer, walked through with numbers.
- 15 min — Stability, simulators, reward hacking. Session 10 trains and compares to a baseline.

## Learning goals
- Explain why a table fails when the state is large or continuous.
- Say what experience replay and a target network are for.
- Name one business reason a firm would rather simulate than explore on live orders.


## 1. The table runs out of rows

Week 4 had 5 stock levels × 3 orders = 15 numbers. Add 100 SKUs, or a price, or a day-of-week, and the table is mostly empty. A neural network is a **shared** guess: similar states share weights, so an unseen state still gets a Q-value. The cost is that the guess can be wrong everywhere at once.

| Method | What it stores | Business reading |
| --- | --- | --- |
| Value-based (DQN) | Q(state, action) | "How good is this order here?" |
| Policy-based | A rule from state to action | "What do we do here?" without scores for the rejected orders |
| Model-based | A learned transition | Useful when a wrong live action is expensive and you can plan in the model |

We stay with DQN this week because students already know Q-learning.


In [ ]:
import numpy as np

# One state, three actions. The network replaces one row of the Q-table.
# Features: stock/4 and yesterday's demand/2. Both sit in [0, 1].


class TinyQ:
    """One hidden layer. Small so the class can see every matrix."""

    def __init__(self, rng):
        self.rng = rng
        self.w1 = rng.normal(0, 0.2, size=(2, 8))
        self.b1 = np.zeros(8)
        self.w2 = rng.normal(0, 0.2, size=(8, 3))
        self.b2 = np.zeros(3)

    def forward(self, x):
        self.x = np.asarray(x, dtype=float)
        self.z1 = self.x @ self.w1 + self.b1
        self.h = np.maximum(0, self.z1)  # ReLU: negative signals do not pass
        self.q = self.h @ self.w2 + self.b2
        return self.q

    def copy_from(self, other):
        self.w1 = other.w1.copy()
        self.b1 = other.b1.copy()
        self.w2 = other.w2.copy()
        self.b2 = other.b2.copy()


online = TinyQ(np.random.default_rng(1))
target = TinyQ(np.random.default_rng(2))
target.copy_from(online)
state = np.array([0.25, 0.5])  # stock 1 of 4, demand 1 of 2
print("Q(state) from the online net", np.round(online.forward(state), 3))
print("Q(state) from the target net", np.round(target.forward(state), 3))
print("They match right after the copy, and then the target is held still on purpose.")


## 2. Replay and a target

If you train only on this morning's order, the network forgets last week (correlated updates). **Experience replay** stores `(state, action, reward, next_state)` and learns from a random sample.

The TD target uses a **second** network that you update only every N steps. If the same network both predicts Q and builds the target, the target moves every time you learn and training chases itself.

## 3. What to tell a manager before Session 10's training run

- Live exploration orders the wrong quantity on real customers. Firms train in a **simulator** built from history, then shadow the policy.
- **Reward hacking:** if the reward is "never stock out" and holding is free in the formula, the net will over-order. Fix the reward, do not celebrate the loss curve.
- A diverging loss is a failed run, not a bold policy. In Session 10 you will compare against a base-stock rule before anyone says "deploy."


## Fill this with me

The two networks match right after the copy. If we then train only the online network, does the target stay equal until we copy again?


In [ ]:
# FILL WITH THE CLASS. yes or no.
target_stays_equal = ""
print("You said:", target_stays_equal or "yes or no, then we will say why the target is frozen")
